# AI Product Intelligence System
## Gen AI Bootcamp — Day 2 Challenge

### Tasks
1. **Smart Product Recommendation Engine** — Recommend complementary products
2. **Unique Product Catalog Creation** — Deduplicate near-identical products using embeddings
3. **Reverse Product Search** — Text-to-image search using CLIP

### Dataset
Fashion Product Images Small (Kaggle)

---

## Setup & Installation

In [ ]:
!pip install torch torchvision transformers ftfy regex tqdm scikit-learn faiss-cpu pillow matplotlib pandas -q

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from transformers import CLIPTokenizer, CLIPModel
from sklearn.cluster import DBSCAN
import faiss
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Load Dataset

In [ ]:
DATA_DIR = '/kaggle/input/fashion-product-images-small'
if not os.path.exists(DATA_DIR):
    DATA_DIR = './fashion-product-images-small'
if not os.path.exists(DATA_DIR):
    DATA_DIR = './data'

print(f'Data directory: {DATA_DIR}')
print(f'Contents: {os.listdir(DATA_DIR)[:10]}')

In [ ]:
styles_df = pd.read_csv(os.path.join(DATA_DIR, 'styles.csv'), on_bad_lines='skip')
print(f'Dataset shape: {styles_df.shape}')
print(f'Columns: {list(styles_df.columns)}')
styles_df.head()

In [ ]:
def get_image_path(product_id):
    path = os.path.join(DATA_DIR, 'images', f'{product_id}.jpg')
    return path if os.path.exists(path) else None

styles_df['image_path'] = styles_df['id'].apply(get_image_path)
df = styles_df.dropna(subset=['image_path']).reset_index(drop=True)
print(f'Products with images: {len(df)}')
print('\nTop product types:')
print(df['articleType'].value_counts().head(15))

In [ ]:
SAMPLE_SIZE = 2000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f'Working with {len(df)} products')
df[['id', 'productDisplayName', 'articleType', 'baseColour', 'gender']].head(10)

## Load CLIP Model

In [ ]:
print('Loading CLIP model...')
tokenizer = CLIPTokenizer.from_pretrained('openai/clip-vit-base-patch32')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32')
clip_model.eval()
print('CLIP model loaded!')

In [ ]:
def encode_images(image_paths, batch_size=32):
    all_embeddings = []
    valid_indices = []
    from torchvision import transforms
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.48145466, 0.4578275, 0.40821073],
                             [0.26862954, 0.26130258, 0.27577711])
    ])
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        batch_images = []
        batch_valid = []
        for j, path in enumerate(batch_paths):
            try:
                img = Image.open(path).convert('RGB')
                batch_images.append(preprocess(img))
                batch_valid.append(i + j)
            except Exception:
                continue
        if not batch_images:
            continue
        batch_tensor = torch.stack(batch_images)
        with torch.no_grad():
            output = clip_model.get_image_features(batch_tensor)
            emb = output.pooler_output if hasattr(output, 'pooler_output') else output[0]
            emb = emb / emb.norm(dim=-1, keepdim=True)
        all_embeddings.append(emb.cpu().numpy())
        valid_indices.extend(batch_valid)
        if (i // batch_size) % 5 == 0:
            print(f'  Encoded {min(i+batch_size, len(image_paths))}/{len(image_paths)} images')
    return np.vstack(all_embeddings), valid_indices

print(f'Encoding {len(df)} product images with CLIP...')
image_embeddings, valid_idx = encode_images(df['image_path'].tolist())
df = df.iloc[valid_idx].reset_index(drop=True)
print(f'\nEmbedding matrix shape: {image_embeddings.shape}')

In [ ]:
embedding_dim = image_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(image_embeddings.astype('float32'))
print(f'FAISS index built: {faiss_index.ntotal} vectors, dimension {embedding_dim}')

---
## Task 1: Smart Product Recommendation Engine

**Goal:** Recommend complementary products (e.g., running shoes -> sports socks, fitness watch)

**Approach:**
1. **Category Complementarity Rules** — Define which product categories go together
2. **Cross-Category Similarity** — Use CLIP embeddings to find the best match within each complementary category
3. **Ranking** — Score candidates by embedding similarity + complementarity strength

### How Recommendations Are Generated

1. **Complement Map** — A curated dictionary maps each product category (e.g., "Running Shoes") to complementary categories (e.g., "Socks", "Track Pants", "Tshirts"). This encodes fashion domain knowledge about what products are commonly purchased together.

2. **Visual Similarity via CLIP** — For each complementary category, all products in that category are compared against the input product using cosine similarity of their CLIP image embeddings. This ensures the recommended items visually match the style of the input.

3. **Ranking** — Candidates from all complementary categories are ranked by similarity score. The top-k unique products are returned.

4. **Fallback** — If no complement rules exist for a category, random different categories are sampled.

In [ ]:
COMPLEMENT_MAP = {
    'Running Shoes':        ['Sports Shoes', 'Socks', 'Track Pants', 'Tshirts', 'Water Bottle'],
    'Sports Shoes':         ['Socks', 'Track Pants', 'Shorts', 'Tshirts'],
    'Sneakers':             ['Jeans', 'Tshirts', 'Casual Shoes', 'Socks'],
    'Formal Shoes':         ['Trousers', 'Shirts', 'Belts', 'Socks'],
    'Shirts':               ['Trousers', 'Formal Shoes', 'Belts', 'Watches'],
    'Trousers':             ['Shirts', 'Formal Shoes', 'Belts'],
    'Jeans':                ['Tshirts', 'Casual Shoes', 'Sneakers', 'Jackets'],
    'Tshirts':              ['Jeans', 'Shorts', 'Sneakers', 'Sunglasses'],
    'Shorts':               ['Tshirts', 'Slippers', 'Sneakers'],
    'Jackets':              ['Jeans', 'Trousers', 'Sneakers', 'Tshirts'],
    'Hoodies':              ['Jeans', 'Sneakers', 'Tshirts'],
    'Sweaters':             ['Trousers', 'Jeans', 'Formal Shoes'],
    'Watches':              ['Shirts', 'Trousers', 'Formal Shoes', 'Belts'],
    'Belts':                ['Trousers', 'Shirts', 'Formal Shoes'],
    'Sunglasses':           ['Tshirts', 'Shorts', 'Jeans', 'Casual Shoes'],
    'Bags':                 ['Tshirts', 'Jeans', 'Sneakers'],
    'Heels':                ['Dresses', 'Kurtas', 'Handbags'],
    'Flats':                ['Jeans', 'Kurtas', 'Tshirts'],
    'Dresses':              ['Heels', 'Handbags', 'Earrings'],
    'Kurtas':               ['Trousers', 'Flats', 'Earrings'],
}

ALL_CATEGORIES = df['articleType'].unique().tolist()
print(f'Complement rules defined for {len(COMPLEMENT_MAP)} product types')
print(f'Total unique categories in dataset: {len(ALL_CATEGORIES)}')

In [ ]:
def recommend_complementary(product_idx, top_k=5):
    product = df.iloc[product_idx]
    product_name = product['productDisplayName']
    product_category = product['articleType']
    product_embedding = image_embeddings[product_idx:product_idx+1]
    comp_cats = COMPLEMENT_MAP.get(product_category, [])
    if not comp_cats:
        other_cats = [c for c in ALL_CATEGORIES if c != product_category]
        comp_cats = np.random.choice(other_cats, size=min(5, len(other_cats)), replace=False).tolist()
    candidates = []
    for comp_cat in comp_cats:
        cat_mask = df['articleType'] == comp_cat
        cat_indices = np.where(cat_mask.values)[0]
        if len(cat_indices) == 0:
            continue
        cat_embeddings = image_embeddings[cat_indices].astype('float32')
        similarities = np.dot(cat_embeddings, product_embedding.T).flatten()
        top_local = np.argsort(similarities)[-2:][::-1]
        for local_idx in top_local:
            global_idx = cat_indices[local_idx]
            candidates.append({
                'idx': global_idx,
                'name': df.iloc[global_idx]['productDisplayName'],
                'category': comp_cat,
                'color': df.iloc[global_idx]['baseColour'],
                'similarity': float(similarities[local_idx]),
                'complement_to': product_category,
                'image_path': df.iloc[global_idx]['image_path']
            })
    candidates.sort(key=lambda x: x['similarity'], reverse=True)
    seen = set()
    unique = []
    for c in candidates:
        if c['idx'] not in seen:
            seen.add(c['idx'])
            unique.append(c)
    return {
        'input_name': product_name,
        'input_category': product_category,
        'input_image': product['image_path'],
        'recommendations': unique[:top_k]
    }

In [ ]:
def visualize_recommendations(result):
    recs = result['recommendations']
    n = len(recs) + 1
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 7))
    if n == 1:
        axes = [axes]
    ax = axes[0]
    img = Image.open(result['input_image']).convert('RGB')
    img = img.resize((300, 400), Image.NEAREST)
    ax.imshow(img, interpolation='nearest')
    ax.set_title(f'INPUT\n{result["input_name"][:30]}\n({result["input_category"]})',
                 fontsize=11, fontweight='bold', color='#0A66C2')
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('#0A66C2')
        spine.set_linewidth(4)
    for i, rec in enumerate(recs):
        ax = axes[i + 1]
        img = Image.open(rec['image_path']).convert('RGB')
        img = img.resize((300, 400), Image.NEAREST)
        ax.imshow(img, interpolation='nearest')
        pct = rec['similarity'] * 100
        ax.set_title(f'#{i+1} [{rec["category"]}]\n{rec["name"][:28]}\n{pct:.0f}% match', fontsize=10)
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color('#057642')
            spine.set_linewidth(2)
    fig.suptitle('Complementary Product Recommendations', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('task1_recommendations.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: task1_recommendations.png')

In [ ]:
print('Sample products by category:')
for cat in ['Running Shoes', 'Shirts', 'Jeans', 'Watches', 'Sneakers']:
    subset = df[df['articleType'] == cat]
    if len(subset) > 0:
        sample = subset.iloc[0]
        idx = subset.index[0]
        print(f'  [{cat}] idx={idx}: {sample["productDisplayName"]}')

In [ ]:
running_mask = df['articleType'] == 'Running Shoes'
if running_mask.sum() > 0:
    test_idx = df[running_mask].index[0]
    result = recommend_complementary(test_idx, top_k=5)
    print(f'Input: {result["input_name"]} ({result["input_category"]})')
    print(f'\nTop {len(result["recommendations"])} complementary products:')
    for i, rec in enumerate(result['recommendations']):
        print(f'  {i+1}. [{rec["category"]}] {rec["name"]} (similarity: {rec["similarity"]:.3f})')
    visualize_recommendations(result)

In [ ]:
shirt_mask = df['articleType'] == 'Shirts'
if shirt_mask.sum() > 0:
    test_idx = df[shirt_mask].index[0]
    result = recommend_complementary(test_idx, top_k=5)
    print(f'Input: {result["input_name"]} ({result["input_category"]})')
    for i, rec in enumerate(result['recommendations']):
        print(f'  {i+1}. [{rec["category"]}] {rec["name"]} (similarity: {rec["similarity"]:.3f})')
    visualize_recommendations(result)

In [ ]:
jeans_mask = df['articleType'] == 'Jeans'
if jeans_mask.sum() > 0:
    test_idx = df[jeans_mask].index[0]
    result = recommend_complementary(test_idx, top_k=5)
    print(f'Input: {result["input_name"]} ({result["input_category"]})')
    for i, rec in enumerate(result['recommendations']):
        print(f'  {i+1}. [{rec["category"]}] {rec["name"]} (similarity: {rec["similarity"]:.3f})')
    visualize_recommendations(result)

---
## Task 2: Unique Product Catalog Creation

**Goal:** Identify duplicate/near-duplicate products and create a clean catalog of unique products.

**Approach:**
1. Encode all products with CLIP
2. Use DBSCAN clustering on embedding space to group similar products
3. Select one representative from each cluster

### How Deduplication Works

1. **CLIP Encoding** — Each product image is encoded into a 512-dimensional embedding using CLIP. Visually identical products produce very similar embeddings.

2. **DBSCAN Clustering** — DBSCAN groups products that are within a cosine distance threshold (eps). Products with cosine distance < eps are considered duplicates. DBSCAN automatically determines the number of clusters and handles outliers (noise).

3. **Representative Selection** — One product from each cluster is chosen as the catalog entry, producing a clean deduplicated catalog.

In [ ]:
print('Finding optimal DBSCAN parameters...')
from sklearn.neighbors import NearestNeighbors
k = 5
nn = NearestNeighbors(n_neighbors=k+1, metric='cosine')
nn.fit(image_embeddings)
distances, _ = nn.kneighbors(image_embeddings)
k_distances = np.sort(distances[:, k])[::-1]

plt.figure(figsize=(10, 5))
plt.plot(k_distances)
plt.xlabel('Points (sorted)')
plt.ylabel(f'Distance to {k}th neighbor')
plt.title('DBSCAN eps Selection')
plt.grid(True, alpha=0.3)
plt.savefig('task2_distance_curve.png', dpi=150)
plt.show()
print(f'Distance stats:')
print(f'  25th percentile: {np.percentile(k_distances, 25):.4f}')
print(f'  50th percentile: {np.percentile(k_distances, 50):.4f}')
print(f'  75th percentile: {np.percentile(k_distances, 75):.4f}')
print(f'  90th percentile: {np.percentile(k_distances, 90):.4f}')

In [ ]:
results = {}
for eps in [0.10, 0.12, 0.14, 0.16, 0.18, 0.20]:
    db = DBSCAN(eps=eps, min_samples=3, metric='cosine', n_jobs=-1)
    labels = db.fit_predict(image_embeddings)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    results[eps] = (n_clusters, n_noise)
    print(f'eps={eps:.2f} -> {n_clusters} clusters, {n_noise} noise')

best_eps = max(results, key=lambda e: results[e][0] if 10 <= results[e][0] <= 100 else 0)
if results[best_eps][0] < 5:
    best_eps = max(results, key=lambda e: results[e][0])
print(f'\nBest eps: {best_eps} ({results[best_eps][0]} clusters)')

db = DBSCAN(eps=best_eps, min_samples=3, metric='cosine', n_jobs=-1)
catalog_labels = db.fit_predict(image_embeddings)
df['cluster'] = catalog_labels

n_clusters = len(set(catalog_labels)) - (1 if -1 in catalog_labels else 0)
n_noise = (catalog_labels == -1).sum()
print(f'Final: {n_clusters} clusters, {n_noise} noise points ({n_noise/len(df)*100:.1f}%)')
print('\nCluster sizes:')
for cid in sorted(set(catalog_labels)):
    if cid == -1:
        print(f'  Noise: {(catalog_labels == -1).sum()}')
    else:
        members = df[df['cluster'] == cid]
        cats = members['masterCategory'].value_counts().head(3).index.tolist()
        print(f'  Cluster {cid}: {len(members)} items | {", ".join(cats)}')

In [ ]:
def extract_representatives(df):
    reps = df.groupby('cluster').first().reset_index()
    return reps

unique_catalog = extract_representatives(df)
print(f'Unique catalog: {len(unique_catalog)} products (from {len(df)} originals)')
print(f'Deduplication rate: {100 * (1 - len(unique_catalog)/len(df)):.1f}%')
unique_catalog[['id', 'productDisplayName', 'articleType', 'baseColour', 'cluster']].head(15)

In [ ]:
clusters = sorted(set(catalog_labels))
clusters = [c for c in clusters if c != -1]
n_show = min(8, len(clusters))

if n_show > 0:
    fig, axes = plt.subplots(n_show, 6, figsize=(18, 3 * n_show))
    if n_show == 1:
        axes = [axes]
    for row, cid in enumerate(clusters[:n_show]):
        members = df[df['cluster'] == cid]
        cats = members['masterCategory'].unique()
        ax = axes[row]
        ax[0].text(0.5, 0.5, f'Cluster {cid}\n{len(members)} items\n{", ".join(cats[:3])}',
                   ha='center', va='center', fontsize=10, fontweight='bold',
                   bbox=dict(boxstyle='round', facecolor='#E8F0FE'))
        ax[0].axis('off')
        samples = members.sample(min(5, len(members)), random_state=42)
        for col, (_, item) in enumerate(samples.iterrows()):
            img = Image.open(item['image_path']).convert('RGB')
            img = img.resize((200, 267), Image.NEAREST)
            ax[col + 1].imshow(img, interpolation='nearest')
            ax[col + 1].set_title(item['productDisplayName'][:20], fontsize=8)
            ax[col + 1].axis('off')
    fig.suptitle('Product Catalog Clusters (DBSCAN)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('task2_catalog_clusters.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: task2_catalog_clusters.png')

---
## Task 3: Reverse Product Search

**Goal:** Search products using natural language text descriptions.

**Approach:**
1. Encode the text query with CLIP's text encoder
2. Compare against all image embeddings using cosine similarity
3. Return top-k most similar products

This works because CLIP learns a shared embedding space for images and text.

### How Reverse Search Works

1. **Text Encoding** — The user's text query (e.g., "blue casual shirt") is encoded using CLIP's text encoder into a 512-dimensional vector in the same embedding space as the images.

2. **FAISS Index** — All product image embeddings are stored in a FAISS index for fast approximate nearest neighbor search.

3. **Similarity Search** — The text embedding is compared against all image embeddings using inner product (equivalent to cosine similarity for normalized vectors). The top-k most similar products are returned.

4. **Why it works** — CLIP is trained on 400M image-text pairs to align visual and textual representations. A text query like "blue casual shirt" lands near images of blue casual shirts in the shared embedding space.

In [ ]:
def search_by_text(query, k=5):
    inputs = tokenizer([query], padding=True, return_tensors='pt')
    with torch.no_grad():
        output = clip_model.get_text_features(**inputs)
        text_emb = output.pooler_output if hasattr(output, 'pooler_output') else output[0]
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)
    text_emb = text_emb.cpu().numpy().astype('float32')
    D, I = faiss_index.search(text_emb, k)
    results = []
    for rank, (idx, score) in enumerate(zip(I[0], D[0])):
        p = df.iloc[idx]
        results.append({
            'rank': rank + 1,
            'name': p['productDisplayName'],
            'category': p['articleType'],
            'color': p.get('baseColour', 'N/A'),
            'similarity': float(score),
            'image_path': p['image_path'],
        })
    return results

In [ ]:
def visualize_search(query, results):
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 6))
    if n == 1:
        axes = [axes]
    for i, r in enumerate(results):
        ax = axes[i]
        img = Image.open(r['image_path']).convert('RGB')
        img = img.resize((300, 400), Image.NEAREST)
        ax.imshow(img, interpolation='nearest')
        pct = r['similarity'] * 100
        ax.set_title(f'#{r["rank"]} [{r["category"]}]\n{r["name"][:25]}\n{r["color"]} | {pct:.0f}%', fontsize=10)
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color('#057642')
            spine.set_linewidth(2)
    fig.suptitle(f'Reverse Search: "{query}"', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('task3_search_result.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: task3_search_result.png')

In [ ]:
queries = [
    'red running shoes for men',
    'floral summer dress',
    'black leather formal shoes',
    'blue denim jeans',
]
for q in queries:
    print(f'\n{"="*60}')
    print(f'QUERY: "{q}"')
    print('=' * 60)
    results = search_by_text(q, k=5)
    for r in results:
        print(f'  #{r["rank"]} {r["name"][:35]} | {r["category"]} | {r["color"]} | {r["similarity"]:.3f}')
    visualize_search(q, results)

---
## Summary

### Architecture
```
CLIP Model (openai/clip-vit-base-patch32)
    +-- Image Encoder: Product images -> 512-dim embeddings
    +-- Text Encoder:  Text queries   -> 512-dim embeddings
    +-- Shared Space:  Images & text in same vector space

Task 1: Complementary Recommendations
    Category Rules + CLIP Visual Similarity -> Ranked complementary products

Task 2: Unique Catalog
    CLIP Embeddings + DBSCAN Clustering -> Deduplicated catalog

Task 3: Reverse Search
    CLIP Text Encoding + FAISS Index -> Text-to-image retrieval
```

### Results Summary

| Task | Method | Key Technology |
|------|--------|----------------|
| 1. Complementary Recs | Category rules + embedding similarity | CLIP image embeddings |
| 2. Unique Catalog | DBSCAN clustering on CLIP embeddings | Cosine distance + DBSCAN |
| 3. Reverse Search | CLIP text-to-image retrieval | CLIP text encoder + FAISS |